In [ ]:
"""
Full Preprocessing Pipeline  (Steps 1-6 combined)
=====================================================
Input  : Movies_Reviews_modified_version1.csv   (raw dataset)
Outputs: pipeline_before.csv                    (copy of raw input)
         movies_reviews_train_final.csv          (modeling-ready train set)
         movies_reviews_test_final.csv           (modeling-ready test set)
         tfidf_vectorizer.joblib                 (fitted TF-IDF vectorizer)
Charts : chart_step1_cleaning.png
         chart_step2_outliers.png
         chart_step3_top_words.png
         chart_step4_genre_frequency.png
         chart_step4_emotion_distribution.png
         chart_step5_tfidf_top_terms.png
         chart_step6_svd_scatter.png

Run from any working directory -- all paths are resolved relative to this script.
"""

# -- Standard library --
import ast, re, sys
from collections import Counter
from pathlib import Path

# -- Third-party --
import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

plt.rcParams["font.family"] = "DejaVu Sans"

# ==============================================================================
# PATHS  -- everything lives next to this script
# ==============================================================================
HERE = Path(__file__).resolve().parent

RAW_INPUT_FILE = HERE / "Movies_Reviews_modified_version1.csv"

BEFORE_CSV     = HERE / "pipeline_before.csv"
TRAIN_OUT      = HERE / "movies_reviews_train_final.csv"
TEST_OUT       = HERE / "movies_reviews_test_final.csv"
VECTORIZER_OUT = HERE / "tfidf_vectorizer.joblib"

CHART_S1        = HERE / "chart_step1_cleaning.png"
CHART_S2        = HERE / "chart_step2_outliers.png"
CHART_S3        = HERE / "chart_step3_top_words.png"
CHART_S4_GENRE  = HERE / "chart_step4_genre_frequency.png"
CHART_S4_EMOT   = HERE / "chart_step4_emotion_distribution.png"
CHART_S5        = HERE / "chart_step5_tfidf_top_terms.png"
CHART_S6        = HERE / "chart_step6_svd_scatter.png"

# ==============================================================================
# STEP 5 / 6 CONFIG
# ==============================================================================
MAX_TFIDF_FEATURES = 1000    # Cap vocabulary to avoid exploding column counts / memory
NGRAM_RANGE        = (1, 2)  # Use both unigrams ("good") and bigrams ("very good")
K_BEST             = 500     # SelectKBest: how many TF-IDF features to keep
SVD_COMPONENTS     = 2       # SVD components for 2D visualisation only -- NOT exported
POSITIVE_THRESHOLD = 7.0     # Ratings >= this => Positive
NEGATIVE_THRESHOLD = 5.0     # Ratings <  this => Negative  (between => Neutral)

# ==============================================================================
# SHARED HELPERS
# ==============================================================================

def parse_genres(cell) -> list:
    """
    Safely convert a genres cell from its CSV string form to a Python list.
    Returns [] for NaN, already-a-list, or unparseable strings.
    """
    if pd.isna(cell):
        return []
    if isinstance(cell, list):
        return cell
    try:
        parsed = ast.literal_eval(str(cell))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

def most_common_genre_list(genre_series: pd.Series):
    """
    Given a Series of genre lists (one movie's rows), return the most frequent
    non-empty genre list.  Returns None if every entry is [].
    """
    non_empty = [tuple(g) for g in genre_series if len(g) > 0]
    if not non_empty:
        return None
    mode_tuple, _ = Counter(non_empty).most_common(1)[0]
    return list(mode_tuple)

# ==============================================================================
# STEP 1 -- Missing / Placeholder Data Handling + Deduplication
# ==============================================================================

UNKNOWN_MOVIE_PLACEHOLDER = "Unknown"
UNKNOWN_GENRE_FALLBACK    = ["Unknown genre"]

def step1_clean(df: pd.DataFrame):
    print("=" * 70)
    print("STEP 1: Missing / Placeholder Data Handling + Deduplication")
    print("=" * 70)

    df["genres"] = df["genres"].apply(parse_genres)

    before_unknown_movie = (df["movie_name"] == UNKNOWN_MOVIE_PLACEHOLDER).sum()
    before_empty_genres  = df["genres"].apply(len).eq(0).sum()
    before_dedup_total   = len(df)
    before_dup_reviews   = df["Reviews"].duplicated(keep=False).sum()

    print(f"\n[Before] 'Unknown' movie_name rows : {before_unknown_movie:,}")
    print(f"[Before] Empty genres [] rows       : {before_empty_genres:,}")
    print(f"[Before] Total rows                 : {before_dedup_total:,}")
    print(f"[Before] Rows with duplicate Reviews: {before_dup_reviews:,}")

    df["is_unknown_movie_name"] = df["movie_name"] == UNKNOWN_MOVIE_PLACEHOLDER

    movie_mode_genres = (
        df.groupby("movie_name")["genres"]
        .apply(most_common_genre_list)
        .to_dict()
    )

    def apply_movie_mode_genre(row):
        canonical = movie_mode_genres.get(row["movie_name"])
        if canonical is not None:
            return canonical
        return row["genres"]

    df["genres"] = df.apply(apply_movie_mode_genre, axis=1)

    n_filled = before_empty_genres - df["genres"].apply(len).eq(0).sum()
    print(f"\n[Genre] Empty-genre rows filled via mode propagation: {n_filled:,}")

    still_empty_mask = df["genres"].apply(len).eq(0)
    n_orphaned = still_empty_mask.sum()

    if n_orphaned > 0:
        df.loc[still_empty_mask, "genres"] = df.loc[still_empty_mask, "genres"].apply(
            lambda _: list(UNKNOWN_GENRE_FALLBACK)
        )
        print(f"[Impute] {n_orphaned:,} rows had no genre data -> imputed as {UNKNOWN_GENRE_FALLBACK}.")

    after_empty_genres  = df["genres"].apply(len).eq(0).sum()
    after_unknown_movie = df["is_unknown_movie_name"].sum()

    dup_reviews_before = df["Reviews"].duplicated(keep=False).sum()
    df = df.drop_duplicates(subset=["Reviews"], keep="first")
    dup_reviews_after = df["Reviews"].duplicated(keep=False).sum()

    print(f"[Dedup] Rows removed     : {before_dedup_total - len(df):,}")
    print(f"[Dedup] Rows remaining   : {len(df):,}")

    # -- Chart --
    COLORS_BEFORE = "#E07B54"
    COLORS_AFTER  = "#4C9BE8"
    BAR_WIDTH     = 0.35

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle("Step 1 -- Data Quality: Before vs. After Cleaning",
                 fontsize=14, fontweight="bold", y=1.01)

    labels1        = ["'Unknown'\nmovie_name", "Empty\ngenres []"]
    before_counts1 = [before_unknown_movie, before_empty_genres]
    after_counts1  = [after_unknown_movie,  after_empty_genres]
    x1 = range(len(labels1))
    bars_b1 = ax1.bar([i - BAR_WIDTH / 2 for i in x1], before_counts1, BAR_WIDTH,
                      label="Before", color=COLORS_BEFORE, edgecolor="white")
    bars_a1 = ax1.bar([i + BAR_WIDTH / 2 for i in x1], after_counts1, BAR_WIDTH,
                      label="After",  color=COLORS_AFTER,  edgecolor="white")
    ax1.set_xticks(list(x1)); ax1.set_xticklabels(labels1, fontsize=11)
    ax1.set_ylabel("Row count", fontsize=11)
    ax1.set_title("Missing / Placeholder Data", fontsize=12, fontweight="bold")
    ax1.legend(fontsize=10); ax1.spines[["top","right"]].set_visible(False)
    max1 = max(before_counts1) if max(before_counts1) > 0 else 1
    for bar in list(bars_b1) + list(bars_a1):
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2, h + max1 * 0.02,
                 f"{int(h):,}", ha="center", va="bottom", fontsize=9, color="#333")

    labels2        = ["Rows with\nDuplicate Reviews"]
    before_counts2 = [dup_reviews_before]
    after_counts2  = [dup_reviews_after]
    x2 = range(len(labels2))
    bars_b2 = ax2.bar([i - BAR_WIDTH / 2 for i in x2], before_counts2, BAR_WIDTH,
                      label="Before", color=COLORS_BEFORE, edgecolor="white")
    bars_a2 = ax2.bar([i + BAR_WIDTH / 2 for i in x2], after_counts2, BAR_WIDTH,
                      label="After",  color=COLORS_AFTER,  edgecolor="white")
    ax2.set_xticks(list(x2)); ax2.set_xticklabels(labels2, fontsize=11)
    ax2.set_ylabel("Row count", fontsize=11)
    ax2.set_title("Duplicate Reviews (identical text)", fontsize=12, fontweight="bold")
    ax2.legend(fontsize=10); ax2.spines[["top","right"]].set_visible(False)
    max2 = max(before_counts2) if max(before_counts2) > 0 else 1
    for bar in list(bars_b2) + list(bars_a2):
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2, h + max2 * 0.02,
                 f"{int(h):,}", ha="center", va="bottom", fontsize=9, color="#333")

    plt.tight_layout()
    plt.savefig(CHART_S1, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n[Chart] Saved -> {CHART_S1.name}")
    print("[Done]  Step 1 complete.\n")
    return df

# ==============================================================================
# STEP 2 -- Outlier Detection & Handling (Review Length)
# ==============================================================================

def cap_review_text(text: str, max_len: int) -> str:
    text = str(text)
    return text[:max_len] if len(text) > max_len else text

def step2_outliers(df: pd.DataFrame):
    print("=" * 70)
    print("STEP 2: Outlier Detection & Handling (Review Length)")
    print("=" * 70)

    df["review_length"] = df["Reviews"].astype(str).apply(len)
    
    print(f"\n[Stats] Review Length (Characters):")
    print(f"        Min : {df['review_length'].min():,}")
    print(f"        Max : {df['review_length'].max():,}")
    print(f"        Mean: {df['review_length'].mean():,.1f}")

    length_before = df["review_length"].copy()

    Q1  = df["review_length"].quantile(0.25)
    Q3  = df["review_length"].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers_before = ((df["review_length"] < lower_bound) | (df["review_length"] > upper_bound)).sum()
    print(f"\n[IQR]   Outliers flagged (Before): {n_outliers_before:,}")

    upper_bound_int = int(upper_bound)
    over_upper_mask = df["review_length"] > upper_bound
    df.loc[over_upper_mask, "Reviews"] = df.loc[over_upper_mask, "Reviews"].apply(
        lambda t: cap_review_text(t, upper_bound_int)
    )
    df["review_length"] = df["Reviews"].astype(str).apply(len)

    n_outliers_after = ((df["review_length"] < lower_bound) | (df["review_length"] > upper_bound)).sum()
    print(f"[Fix]   Capped {over_upper_mask.sum():,} long reviews at {upper_bound_int:,} characters.")
    print(f"[IQR]   Outliers remaining (After): {n_outliers_after:,}")

    # -- Chart --
    COLORS_BEFORE = "#E07B54"
    COLORS_AFTER  = "#4C9BE8"

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Step 2 -- Review Length Outliers: Before vs. After Capping",
                 fontsize=15, fontweight="bold", y=0.98)

    axes[0, 0].boxplot(length_before, orientation="horizontal", patch_artist=True,
                       boxprops=dict(facecolor=COLORS_BEFORE, color='black'),
                       medianprops=dict(color='black', linewidth=1.5))
    axes[0, 0].set_title("Review Length (Before)", fontweight="bold")
    axes[0, 0].set_xlabel("Character Count"); axes[0, 0].set_yticks([])

    axes[0, 1].boxplot(df["review_length"], orientation="horizontal", patch_artist=True,
                       boxprops=dict(facecolor=COLORS_AFTER, color='black'),
                       medianprops=dict(color='black', linewidth=1.5))
    axes[0, 1].set_title("Review Length (After Capping)", fontweight="bold")
    axes[0, 1].set_xlabel("Character Count"); axes[0, 1].set_yticks([])

    axes[1, 0].hist(length_before, bins=60, color=COLORS_BEFORE, edgecolor='white', alpha=0.9)
    axes[1, 0].axvline(upper_bound, color="black", linestyle="--", linewidth=2,
                       label=f"Upper Bound ({upper_bound_int})")
    axes[1, 0].set_title("Distribution (Before)", fontweight="bold")
    axes[1, 0].set_xlabel("Character Count"); axes[1, 0].set_ylabel("Frequency")
    axes[1, 0].legend()

    axes[1, 1].hist(df["review_length"], bins=60, color=COLORS_AFTER, edgecolor='white', alpha=0.9)
    axes[1, 1].axvline(upper_bound, color="black", linestyle="--", linewidth=2,
                       label=f"Upper Bound ({upper_bound_int})")
    axes[1, 1].set_title("Distribution (After Capping)", fontweight="bold")
    axes[1, 1].set_xlabel("Character Count"); axes[1, 1].set_ylabel("Frequency")
    axes[1, 1].legend()

    for ax in axes.flat:
        ax.spines[["top","right"]].set_visible(False)
        ax.grid(axis='x', linestyle='--', alpha=0.4)

    plt.tight_layout(pad=2.0)
    plt.savefig(CHART_S2, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n[Chart] Saved -> {CHART_S2.name}")
    print("[Done]  Step 2 complete.\n")
    return df

# ==============================================================================
# STEP 3 -- Text Normalization
# ==============================================================================

def _ensure_nltk_resources():
    required = ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]
    for resource in required:
        try:
            if "punkt" in resource:
                nltk.data.find(f"tokenizers/{resource}")
            else:
                nltk.data.find(f"corpora/{resource}")
        except LookupError:
            print(f"        Downloading NLTK resource: '{resource}'...")
            try:
                nltk.download(resource, quiet=True)
            except Exception as e:
                pass

def step3_normalize(df: pd.DataFrame):
    print("=" * 70)
    print("STEP 3: Text Normalization")
    print("=" * 70)

    print("\n[Setup] Checking NLTK resources...")
    _ensure_nltk_resources()

    from nltk.corpus import stopwords as nltk_sw
    from nltk.stem import WordNetLemmatizer
    from nltk.tokenize import word_tokenize

    stop_words = set(nltk_sw.words("english"))
    lemmatizer = WordNetLemmatizer()
    print("[Setup] NLTK resources ready.")

    def normalize_text(text: str) -> str:
        text   = str(text).lower()
        text   = re.sub(r"[^a-z\s]", " ", text)
        tokens = word_tokenize(text)
        tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
        return " ".join(tokens)

    print(f"\n[Norm]  Running normalization on {len(df):,} rows...")
    df["Reviews_cleaned"] = df["Reviews"].apply(normalize_text)
    print(f"[Norm]  Done. Created column: 'Reviews_cleaned'.")

    def get_top_words(text_series, n=20, filter_stopwords=False):
        counter = Counter()
        for text in text_series.astype(str):
            words = re.findall(r"[a-z]+", text.lower())
            if filter_stopwords:
                words = [w for w in words if w not in stop_words and len(w) > 1]
            counter.update(words)
        return counter.most_common(n)

    before_top = get_top_words(df["Reviews"], n=20)
    after_top  = get_top_words(df["Reviews_cleaned"], n=20)

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle("Step 3 -- Text Normalization: Top 20 Words",
                 fontsize=16, fontweight="bold", y=0.98)

    words_b, counts_b = zip(*before_top)
    axes[0].barh(words_b[::-1], counts_b[::-1], color="#E07B54", edgecolor="black")
    axes[0].set_title("Before Cleaning (Raw Text)", fontweight="bold")
    axes[0].set_xlabel("Frequency")
    axes[0].spines[["top","right"]].set_visible(False)
    axes[0].grid(axis="x", linestyle="--", alpha=0.5)

    words_a, counts_a = zip(*after_top)
    axes[1].barh(words_a[::-1], counts_a[::-1], color="#4C9BE8", edgecolor="black")
    axes[1].set_title("After Cleaning (No Stopwords, Lemmatized)", fontweight="bold")
    axes[1].set_xlabel("Frequency")
    axes[1].spines[["top","right"]].set_visible(False)
    axes[1].grid(axis="x", linestyle="--", alpha=0.5)

    plt.tight_layout(pad=2.0)
    plt.savefig(CHART_S3, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Chart] Saved -> {CHART_S3.name}")
    print("[Done]  Step 3 complete.\n")
    return df

# ==============================================================================
# STEP 4 -- Categorical Encoding
# ==============================================================================

def step4_encode(df: pd.DataFrame):
    print("=" * 70)
    print("STEP 4: Categorical Encoding")
    print("=" * 70)

    mlb = MultiLabelBinarizer()
    genre_dummies = mlb.fit_transform(df["genres"])
    genre_classes = [f"genre_{c}" for c in mlb.classes_]
    genre_df = pd.DataFrame(genre_dummies, columns=genre_classes, index=df.index)
    df = pd.concat([df, genre_df], axis=1)
    print(f"\n[Encode] {len(genre_classes)} binary genre columns created.")

    emotion_dummies = pd.get_dummies(df["emotion"], prefix="emotion", dtype=int)
    df = pd.concat([df, emotion_dummies], axis=1)
    print(f"[Encode] {len(emotion_dummies.columns)} binary emotion columns created.")

    print(f"\n[Stats] Emotion Class Distribution:")
    emotion_counts = df["emotion"].value_counts()
    total_rows = len(df)

    for emotion, count in emotion_counts.items():
        pct = (count / total_rows) * 100
        flag = "  <-- [FLAG] Minority class (<5%)" if pct < 5.0 else ""
        print(f"        - {emotion:<12}: {count:>6,} ({pct:>5.2f}%){flag}")

    # -- Genre chart --
    genre_sums = genre_df.sum().sort_values(ascending=True)
    plt.figure(figsize=(10, 8))
    clean_labels = [l.replace("genre_", "") for l in genre_sums.index]
    bars = plt.barh(clean_labels, genre_sums.values, color="mediumpurple", edgecolor="black")
    plt.title("Step 4 -- Genre Frequency Distribution", fontsize=14, fontweight="bold")
    plt.xlabel("Number of Reviews")
    plt.gca().spines[["top","right"]].set_visible(False)
    plt.grid(axis="x", linestyle="--", alpha=0.5)
    for bar in bars:
        plt.text(bar.get_width() + (max(genre_sums.values) * 0.01),
                 bar.get_y() + bar.get_height() / 2,
                 f"{int(bar.get_width()):,}", va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(CHART_S4_GENRE, dpi=150)
    plt.close()
    print(f"\n[Chart] Saved -> {CHART_S4_GENRE.name}")

    # -- Emotion chart --
    threshold_5pct = total_rows * 0.05
    plt.figure(figsize=(9, 6))
    bars2 = plt.bar(emotion_counts.index, emotion_counts.values, color="coral", edgecolor="black")
    plt.title("Step 4 -- Emotion Class Distribution", fontsize=14, fontweight="bold")
    plt.ylabel("Number of Reviews")
    plt.xticks(rotation=45, ha="right")
    plt.gca().spines[["top","right"]].set_visible(False)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.axhline(threshold_5pct, color="red", linestyle="--", linewidth=1.5,
                label="5% Minority Threshold")
    plt.legend()
    for bar in bars2:
        plt.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + (max(emotion_counts.values) * 0.02),
                 f"{int(bar.get_height()):,}", ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig(CHART_S4_EMOT, dpi=150)
    plt.close()
    print(f"[Chart] Saved -> {CHART_S4_EMOT.name}")
    print("[Done]  Step 4 complete.\n")
    return df

# ==============================================================================
# STEP 5 -- Feature Engineering (TF-IDF Vectorization)
# ==============================================================================

def step5_tfidf(df: pd.DataFrame):
    print("=" * 70)
    print("STEP 5: Feature Engineering (TF-IDF Vectorization)")
    print("=" * 70)

    initial_len = len(df)
    df = df.dropna(subset=["Reviews_cleaned"])
    if len(df) < initial_len:
        print(f"[Warn]  Dropped {initial_len - len(df)} rows with NaN cleaned reviews.")

    print(f"\n[Split] 80/20 train-test split (stratified by emotion)...")
    try:
        df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["emotion"])
        print("[Split] Stratified split successful.")
    except ValueError:
        print("[Split] Stratified split failed (rare classes). Falling back to random split.")
        df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

    print(f"        Train: {len(df_train):,}  |  Test: {len(df_test):,}")

    print(f"\n[TFIDF] Fitting TfidfVectorizer on training data only...")
    vectorizer = TfidfVectorizer(
        max_features=MAX_TFIDF_FEATURES,
        ngram_range=NGRAM_RANGE,
        stop_words='english'
    )
    X_train_tfidf = vectorizer.fit_transform(df_train["Reviews_cleaned"])
    X_test_tfidf  = vectorizer.transform(df_test["Reviews_cleaned"])

    feature_names = vectorizer.get_feature_names_out()
    tfidf_cols    = [f"tfidf_{w.replace(' ', '_')}" for w in feature_names]
    print(f"[TFIDF] Created {len(feature_names)} text features.")

    df_train_tfidf = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf_cols, index=df_train.index)
    df_test_tfidf  = pd.DataFrame(X_test_tfidf.toarray(),  columns=tfidf_cols, index=df_test.index)

    def prepare_final_dataset(orig, tfidf):
        cat_cols  = [c for c in orig.columns if c.startswith("genre_") or c.startswith("emotion_")]
        meta_cols = [c for c in ["Ratings", "movie_name", "review_length"] if c in orig.columns]
        return pd.concat([orig[meta_cols + cat_cols], tfidf], axis=1)

    train_with_tfidf = prepare_final_dataset(df_train, df_train_tfidf)
    test_with_tfidf  = prepare_final_dataset(df_test,  df_test_tfidf)

    joblib.dump(vectorizer, VECTORIZER_OUT)
    print(f"[Export] Saved TF-IDF vectorizer -> {VECTORIZER_OUT.name}")

    # -- Chart --
    pos_mask = df_train["Ratings"] >= 7.0
    neg_mask = df_train["Ratings"] <  5.0
    top20_pos = df_train_tfidf[pos_mask].mean().sort_values(ascending=False).head(20)
    top20_neg = df_train_tfidf[neg_mask].mean().sort_values(ascending=False).head(20)

    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    fig.suptitle("Step 5 -- Top 20 TF-IDF Terms (Training Data Only)",
                 fontsize=16, fontweight="bold", y=0.98)

    axes[0].barh(top20_pos.index[::-1].str.replace('tfidf_', ''),
                 top20_pos.values[::-1], color="mediumseagreen", edgecolor="black")
    axes[0].set_title("Positive Reviews (Rating >= 7.0)", fontweight="bold")
    axes[0].set_xlabel("Mean TF-IDF Score")
    axes[0].spines[["top","right"]].set_visible(False)
    axes[0].grid(axis="x", linestyle="--", alpha=0.5)

    axes[1].barh(top20_neg.index[::-1].str.replace('tfidf_', ''),
                 top20_neg.values[::-1], color="indianred", edgecolor="black")
    axes[1].set_title("Negative Reviews (Rating < 5.0)", fontweight="bold")
    axes[1].set_xlabel("Mean TF-IDF Score")
    axes[1].spines[["top","right"]].set_visible(False)
    axes[1].grid(axis="x", linestyle="--", alpha=0.5)

    plt.tight_layout(pad=2.0)
    plt.savefig(CHART_S5, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Chart] Saved -> {CHART_S5.name}")
    print("[Done]  Step 5 complete.\n")
    return train_with_tfidf, test_with_tfidf

# ==============================================================================
# STEP 6 -- Feature Selection & Dimensionality Reduction
# ==============================================================================

def get_sentiment(rating):
    if pd.isna(rating): return "Unknown"
    if rating >= POSITIVE_THRESHOLD: return "Positive"
    elif rating < NEGATIVE_THRESHOLD: return "Negative"
    else: return "Neutral"

def step6_feature_selection(df_train: pd.DataFrame, df_test: pd.DataFrame):
    print("=" * 70)
    print("STEP 6: Feature Selection & Dimensionality Reduction")
    print("=" * 70)

    tfidf_cols   = [c for c in df_train.columns if c.startswith("tfidf_")]
    genre_cols   = [c for c in df_train.columns if c.startswith("genre_")]
    emotion_cols = [c for c in df_train.columns if c.startswith("emotion_")]
    cat_cols     = genre_cols + emotion_cols
    target_col   = "Ratings"
    meta_cols    = [c for c in ["movie_name", "review_length"] if c in df_train.columns]

    print(f"\n[Cols]  TF-IDF: {len(tfidf_cols)}  Genre: {len(genre_cols)}  Emotion: {len(emotion_cols)}")

    X_train_tfidf = df_train[tfidf_cols].fillna(0)
    X_test_tfidf  = df_test[tfidf_cols].fillna(0)
    y_train       = df_train[target_col]

    print(f"\n[Select] SelectKBest (chi2) keeping top {K_BEST} of {len(tfidf_cols)} TF-IDF features...")
    train_valid_mask = y_train.notna()
    selector = SelectKBest(chi2, k=K_BEST)
    selector.fit(X_train_tfidf[train_valid_mask], y_train[train_valid_mask])

    X_train_selected = selector.transform(X_train_tfidf)
    X_test_selected  = selector.transform(X_test_tfidf)
    selected_feature_names = np.array(tfidf_cols)[selector.get_support()]

    scores    = selector.scores_
    top10_idx = np.argsort(scores)[::-1][:10]
    print(f"\n[Select] Top 10 features by chi2 score:")
    for i in top10_idx:
        print(f"         {tfidf_cols[i].replace('tfidf_', ''):<30} chi2 = {scores[i]:.2f}")

    print(f"\n[SVD]   TruncatedSVD ({SVD_COMPONENTS} components) for 2D visualization only...")
    svd = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=42)
    X_train_2d = svd.fit_transform(X_train_selected)
    explained_var = svd.explained_variance_ratio_
    print(f"[SVD]   Explained Variance: {sum(explained_var)*100:.2f}% total")

    sentiments = df_train[target_col].apply(get_sentiment)
    sentiment_colors = {
        "Positive": "#4C9BE8",
        "Neutral":  "#F0C040",
        "Negative": "#E07B54",
        "Unknown":  "#AAAAAA",
    }

    fig, ax = plt.subplots(figsize=(11, 8))
    for sentiment, color in sentiment_colors.items():
        mask = sentiments == sentiment
        if mask.sum() == 0: continue
        ax.scatter(X_train_2d[mask, 0], X_train_2d[mask, 1],
                   label=f"{sentiment} ({mask.sum():,})",
                   color=color, alpha=0.35, s=8, edgecolors='none')
    ax.set_title(
        f"Step 6 -- TruncatedSVD 2D Projection of TF-IDF Features\n"
        f"(Training data, colored by Sentiment class)\n"
        f"Explained Variance: {sum(explained_var)*100:.1f}%",
        fontsize=13, fontweight="bold"
    )
    ax.set_xlabel("SVD Component 1", fontsize=11)
    ax.set_ylabel("SVD Component 2", fontsize=11)
    ax.legend(title="Sentiment", fontsize=10, title_fontsize=10, markerscale=3)
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(CHART_S6, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Chart] Saved -> {CHART_S6.name}")

    def build_final_df(selected_array, feat_names, orig_df):
        tfidf_df = pd.DataFrame(selected_array, columns=feat_names, index=orig_df.index)
        other_df = orig_df[cat_cols + [target_col] + meta_cols].copy()
        return pd.concat([other_df, tfidf_df], axis=1)

    train_final = build_final_df(X_train_selected, selected_feature_names, df_train)
    test_final  = build_final_df(X_test_selected,  selected_feature_names, df_test)

    train_final.to_csv(TRAIN_OUT, index=False)
    print(f"\n[Export] Saved Train -> {TRAIN_OUT.name} ({len(train_final):,} rows, {len(train_final.columns)} cols)")

    test_final.to_csv(TEST_OUT, index=False)
    print(f"[Export] Saved Test  -> {TEST_OUT.name} ({len(test_final):,} rows, {len(test_final.columns)} cols)")

    print("[Done]  Step 6 complete.\n")

# ==============================================================================
# MAIN
# ==============================================================================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("  FULL PREPROCESSING PIPELINE  (Steps 1-6)")
    print("=" * 70 + "\n")

    if not RAW_INPUT_FILE.exists():
        print(f"[Error] Raw input file not found:\n        {RAW_INPUT_FILE}")
        sys.exit(1)

    print(f"[Load]  Reading raw input: {RAW_INPUT_FILE.name}")
    df_raw = pd.read_csv(RAW_INPUT_FILE)
    print(f"[Load]  Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

    df = step1_clean(df_raw.copy())
    df = step2_outliers(df)
    df = step3_normalize(df)
    df = step4_encode(df)
    df_train_tfidf, df_test_tfidf = step5_tfidf(df)
    step6_feature_selection(df_train_tfidf, df_test_tfidf)

    print("=" * 70)
    print("  PIPELINE COMPLETE")
    print(f"  Train final: {TRAIN_OUT.name}")
    print(f"  Test final : {TEST_OUT.name}")
    print(f"  Vectorizer : {VECTORIZER_OUT.name}")
    print(f"  Charts     : {HERE}")
    print("=" * 70 + "\n")